# 실험 03: 고도화된 최신 Agent (LangGraph & Guardrails)

## 1. 실험 제목과 목표
- **제목**: 실무형 상태 머신(State Machine) 에이전트 구축
- **목표**: 블랙박스 형태의 `AgentExecutor` 대신, 최신 기술인 **LangGraph**를 도입하여 에이전트의 실행 루프를 직접 커스텀합니다. 특히 교안에 언급된 **Guardrails(위험 방지)** 개념을 적용하여, Agent가 마음대로 툴을 실행하기 전에 사용자에게 승인을 받는 **Human-in-the-loop** 아키텍처를 구현합니다.

## 2. 사전 준비
> **주의**: 이 노트북을 실행하려면 `langgraph` 패키지가 필요합니다.
> 콘솔에서 `pip install langgraph`를 실행해 주세요.

## 3. 라이브러리 import

In [1]:
import os
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from typing import TypedDict, Annotated, Sequence
from langchain_core.messages import BaseMessage
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver

## 4. 위험한(?) Tool 준비 및 환경 세팅
결제나 이메일 발송처럼 봇이 마음대로 실행하면 큰일 나는 가상의 툴을 만듭니다.

In [2]:
load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini")

@tool
def send_money(amount: int, to_who: str) -> str:
    """실제로 돈을 송금하는 매우 위험한 툴입니다."""
    return f"[시스템] {to_who}님에게 {amount}원을 성공적으로 송금했습니다!! (통장 잔고 텅텅)"

tools = [send_money]
llm_with_tools = llm.bind_tools(tools)

## 5. LangGraph 노드 및 상태(State) 정의
LangGraph는 상태(State)를 노드 간에 주고받으며 돌아가는 구조입니다.

In [3]:
# 1. 상태(State) 정의: 이 그래프는 '대화 기록(messages)'을 상태로 가집니다.
# add_messages는 기존 리스트에 새 메시지를 누적해서 더하라는 뜻입니다.
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

# 2. 노드 정의 (LLM 두뇌 역할)
def call_model(state: AgentState):
    messages = state['messages']
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]} # 새로운 응답을 상태에 추가

# 3. 노드 정의 (Tool 실행 손발 역할)
tool_node = ToolNode(tools)

## 6. Graph(실행 루프) 조립 및 Guardrails(안전 장치) 부착 ⭐실무 핵심
점과 선을 잇듯이 Agent의 흐름을 통제합니다.

In [4]:
workflow = StateGraph(AgentState)

# 점(Node) 찍기
workflow.add_node("agent", call_model)
workflow.add_node("action", tool_node)

# 선(Edge) 긋기
workflow.set_entry_point("agent") # 시작점

# agent 노드가 끝난 뒤, 툴을 호출해야 하면 'action'으로 가고, 아니면 끝(END)내라는 조건부 라우팅
workflow.add_conditional_edges(
    "agent",
    tools_condition,
    {"tools": "action", END: END}
)

# action 노드(툴 실행)가 끝나면 다시 agent로 돌아가서 요약해라.
workflow.add_edge("action", "agent")

# *** 핵심: Guardrails (Human-in-the-loop) ***
# MemorySaver를 붙여서 상태를 저장하되, 'action'(툴 실행) 노드 직전에 멈추도록(interrupt_before) 설정합니다.
memory = MemorySaver()
app = workflow.compile(checkpointer=memory, interrupt_before=["action"])

## 7. 실험: 사람의 승인을 기다리는 Agent
돈을 보내라고 명령해 봅시다.

In [5]:
print("=== [1. 사용자의 위험한 명령] ===")
config = {"configurable": {"thread_id": "test-thread-1"}}

# 스트리밍 방식으로 실행하여 어디서 멈추는지 관찰합니다.
for event in app.stream({"messages": [("user", "사장님한테 50000원 송금해줘.")]}, config):
    for key, value in event.items():
        print(f"[{key} 노드 실행 완료]")

# 현재 상태 확인
current_state = app.get_state(config)
print("\n=== [2. Agent 일시 정지 상태] ===")
print("다음 실행될 노드:", current_state.next)
print("Agent가 다음 액션을 위해 대기 중입니다. 관리자의 승인이 필요합니다!")

# 사용자가 승인(그냥 None을 넣고 다시 stream 재개)한다고 가정
print("\n=== [3. 관리자 승인! (Human-in-the-loop)] ===")
for event in app.stream(None, config):
    for key, value in event.items():
        print(f"[{key} 노드 실행 완료]")
        if key == "agent":
            print("\n[최종 봇 답변]:", value['messages'][-1].content)

print("\n-> 결과: 무작정 툴을 실행하는 AgentExecutor와 달리, LangGraph를 쓰면 돈이 송금되기 직전('action' 노드 진입 전)에 코드를 멈추고 사람의 컨펌을 받을 수 있는 완벽한 Guardrails를 구축할 수 있습니다.")

=== [1. 사용자의 위험한 명령] ===
[agent 노드 실행 완료]
[__interrupt__ 노드 실행 완료]

=== [2. Agent 일시 정지 상태] ===
다음 실행될 노드: ('action',)
Agent가 다음 액션을 위해 대기 중입니다. 관리자의 승인이 필요합니다!

=== [3. 관리자 승인! (Human-in-the-loop)] ===
[action 노드 실행 완료]
[agent 노드 실행 완료]

[최종 봇 답변]: 사장님에게 50,000원을 성공적으로 송금했습니다! 현재 통장 잔고가 없습니다.

-> 결과: 무작정 툴을 실행하는 AgentExecutor와 달리, LangGraph를 쓰면 돈이 송금되기 직전('action' 노드 진입 전)에 코드를 멈추고 사람의 컨펌을 받을 수 있는 완벽한 Guardrails를 구축할 수 있습니다.


## 8. 결과 해석

1. **State Machine (LangGraph)**: 상태와 노드로 이루어진 이 구조는, 단순히 LLM에게 제어권을 다 맡기는 것을 넘어 **개발자가 파이썬 코드로 안전 장치와 로직을 자유롭게 튜닝**할 수 있게 해줍니다.
2. **Guardrails (Human-in-the-loop)**: AI가 아무리 똑똑해도 DB 삭제, 대량 메일 발송, 결제 같은 기능은 100% 믿고 맡길 수 없습니다. `interrupt_before` 기능을 통해 "초안을 잡아오면, 마지막 결재 도장은 사람이 찍는" 실무적인 아키텍처가 완성되었습니다.

## 9. Notion 실험 로그

### 발견한 문제점 (또는 차이점)
* 기존 `AgentExecutor`는 편하긴 하지만 "내부에서 무슨 일이 일어나는지, 언제 멈춰야 하는지" 제어하기가 매우 힘들었음.
* LangGraph를 써보니, 플로우차트를 그리듯 명확하게 제어할 수 있었고 교안에 언급된 `Guardrails`를 실 코드로 구현할 수 있었음.

### 다음 개선 방향
* 이제 LLM 뼈대(프롬프트, 모델, RAG, 툴, 에이전트)를 모두 마스터함.
* 실무 프로젝트에 적용할 때는 LangGraph를 기반으로 에이전트를 설계하는 것이 유지보수와 안정성 측면에서 압도적으로 좋다는 결론.